In [ ]:
"""
KoBERT Text Classification - ROC Curve & AUC (Test Set)
D (Weathering Condition), S (Strength), F (Discontinuity Spacing) 모두 처리
- KoBERT 모델 로드 후 test set에 대해 softmax 확률 추출
- Per-class ROC curve, Macro/Micro/Weighted AUC 계산
- test set에 샘플이 0인 클래스 자동 처리
- 모든 결과를 CSV로 저장 (OriginLab용)
"""

import torch
from torch import nn
import torch.nn.functional as F_func
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import gluonnlp as nlp
import numpy as np
from tqdm import tqdm, tqdm_notebook
import pandas as pd
from sklearn.model_selection import train_test_split
import random

from kobert.utils import get_tokenizer
from kobert.pytorch_kobert import get_pytorch_kobert_model

from transformers import AdamW
from transformers.optimization import get_cosine_schedule_with_warmup

import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, auc, roc_auc_score
)
from sklearn.preprocessing import label_binarize
import os

plt.rcParams['font.family'] = 'Arial'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bertmodel, vocab = get_pytorch_kobert_model()

def fix_seed(random_seed):
    torch.manual_seed(random_seed)
    torch.cuda.manual_seed(random_seed)
    torch.cuda.manual_seed_all(random_seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(random_seed)
    random.seed(random_seed)

random_seed = 12
fix_seed(random_seed)

# ============================================================
# Hyperparameters (공통)
# ============================================================
max_len = 512
batch_size = 16
warmup_ratio = 0.1
num_epochs = 8
max_grad_norm = 1
log_interval = 200
learning_rate = 5e-5
num_workers = 0

DATA_DIR = 'data'              # Labeled_data_UTF8_{D,S,F}.csv
MODEL_DIR = 'trained_model'     # {D,S,F}/best_model_lr5e-05_bs16_epoch8.pt
OUT_DIR = 'results'

# ============================================================
# Dataset Class (공통)
# ============================================================
class BERTDataset(Dataset):
    def __init__(self, dataset, sent_idx, label_idx, bert_tokenizer, max_len,
                 pad, pair):
        transform = nlp.data.BERTSentenceTransform(
            bert_tokenizer, max_seq_length=max_len, pad=pad, pair=pair)
        self.sentences = [transform([i[sent_idx]]) for i in dataset]
        self.labels = [np.int32(i[label_idx]) for i in dataset]

    def __getitem__(self, i):
        return (self.sentences[i] + (self.labels[i], ))

    def __len__(self):
        return (len(self.labels))

# ============================================================
# Model Class (공통)
# ============================================================
class BERTClassifier(nn.Module):
    def __init__(self, bert, hidden_size=768, num_classes=5, dr_rate=None, params=None):
        super(BERTClassifier, self).__init__()
        self.bert = bert
        self.dr_rate = dr_rate
        self.classifier = nn.Linear(hidden_size, num_classes)
        if dr_rate:
            self.dropout = nn.Dropout(p=dr_rate)

    def gen_attention_mask(self, token_ids, valid_length):
        attention_mask = torch.zeros_like(token_ids)
        for i, v in enumerate(valid_length):
            attention_mask[i][:v] = 1
        return attention_mask.float()

    def forward(self, token_ids, valid_length, segment_ids):
        attention_mask = self.gen_attention_mask(token_ids, valid_length)
        _, pooler = self.bert(
            input_ids=token_ids,
            token_type_ids=segment_ids.long(),
            attention_mask=attention_mask.float().to(token_ids.device)
        )
        if self.dr_rate:
            out = self.dropout(pooler)
        return self.classifier(out)

tokenizer = get_tokenizer()
tok = nlp.data.BERTSPTokenizer(tokenizer, vocab, lower=False)

# ============================================================
# D, S, F 각각에 대한 설정
# ============================================================
task_configs = [
    {
        'task_name': 'D',
        'task_full_name': 'Weathering Condition',
        'class_labels': ['D1', 'D2', 'D3', 'D4', 'D5'],
        'csv_path': os.path.join(DATA_DIR, 'Labeled_data_UTF8_D.csv'),
        'model_path': os.path.join(MODEL_DIR, 'D', 'best_model_lr{}_bs{}_epoch{}.pt'.format(learning_rate, batch_size, num_epochs)),
        'save_dir': os.path.join(OUT_DIR, 'Text_classification_BERT_D'),
        'cmap': 'Blues',
    },
    {
        'task_name': 'S',
        'task_full_name': 'Strength',
        'class_labels': ['S1', 'S2', 'S3', 'S4', 'S5'],
        'csv_path': os.path.join(DATA_DIR, 'Labeled_data_UTF8_S.csv'),
        'model_path': os.path.join(MODEL_DIR, 'S', 'best_model_lr{}_bs{}_epoch{}.pt'.format(learning_rate, batch_size, num_epochs)),
        'save_dir': os.path.join(OUT_DIR, 'Text_classification_BERT_S'),
        'cmap': 'Reds',
    },
    {
        'task_name': 'F',
        'task_full_name': 'Discontinuity Spacing',
        'class_labels': ['F1', 'F2', 'F3', 'F4', 'F5'],
        'csv_path': os.path.join(DATA_DIR, 'Labeled_data_UTF8_F.csv'),
        'model_path': os.path.join(MODEL_DIR, 'F', 'best_model_lr{}_bs{}_epoch{}.pt'.format(learning_rate, batch_size, num_epochs)),
        'save_dir': os.path.join(OUT_DIR, 'Text_classification_BERT_F'),
        'cmap': 'Greens',
    },
]

# ============================================================
# 메인 루프: D → S → F 순서로 처리
# ============================================================
for config in task_configs:
    task_name = config['task_name']
    task_full_name = config['task_full_name']
    class_labels = config['class_labels']
    num_classes = len(class_labels)
    save_dir = config['save_dir']
    cmap_color = config['cmap']

    print('\n' + '=' * 60)
    print(f'  Processing: {task_name} ({task_full_name})')
    print('=' * 60)

    os.makedirs(save_dir, exist_ok=True)

    # ----------------------------------------------------------
    # Data Loading
    # ----------------------------------------------------------
    text_data = pd.read_csv(config['csv_path'])

    text_data.loc[(text_data['Score'] == 1), 'Score'] = 0
    text_data.loc[(text_data['Score'] == 2), 'Score'] = 1
    text_data.loc[(text_data['Score'] == 3), 'Score'] = 2
    text_data.loc[(text_data['Score'] == 4), 'Score'] = 3
    text_data.loc[(text_data['Score'] == 5), 'Score'] = 4

    data_list = []
    for q, label in zip(text_data['Text'], text_data['Score']):
        data_list.append([q, str(label)])

    dataset_train, dataset_temp = train_test_split(data_list, test_size=0.3, random_state=0)
    dataset_val, dataset_test = train_test_split(dataset_temp, test_size=0.5, random_state=0)

    print(f'  Train: {len(dataset_train)}, Val: {len(dataset_val)}, Test: {len(dataset_test)}')

    data_test = BERTDataset(dataset_test, 0, 1, tok, max_len, True, False)
    test_dataloader = DataLoader(data_test, batch_size=batch_size, num_workers=num_workers)

    # ----------------------------------------------------------
    # Model Loading
    # ----------------------------------------------------------
    model = BERTClassifier(bertmodel, dr_rate=0.5).to(device)
    model.load_state_dict(torch.load(config['model_path'], map_location=device))
    model.eval()

    # ----------------------------------------------------------
    # Inference - Softmax 확률값 추출
    # ----------------------------------------------------------
    all_preds = []
    all_labels = []
    all_probs = []
    softmax_fn = nn.Softmax(dim=1)

    print(f'  Running inference on test set...')
    with torch.no_grad():
        for batch_id, (token_ids, valid_length, segment_ids, label) in enumerate(tqdm_notebook(test_dataloader)):
            token_ids = token_ids.long().to(device)
            segment_ids = segment_ids.long().to(device)
            valid_length = valid_length
            label = label.long().to(device)

            out = model(token_ids, valid_length, segment_ids)
            probs = softmax_fn(out)

            preds = torch.max(out, 1)[1].detach().cpu().numpy()
            labels_np = label.detach().cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(labels_np)
            all_probs.extend(probs.detach().cpu().numpy())

    all_labels_arr = np.array(all_labels)
    all_preds_arr = np.array(all_preds)
    all_probs_arr = np.array(all_probs)

    test_acc = (all_preds_arr == all_labels_arr).sum() / len(all_labels_arr)
    print(f'  Test Accuracy: {test_acc:.4f}')

    # ----------------------------------------------------------
    # 클래스별 샘플 수 확인
    # ----------------------------------------------------------
    class_sample_counts = {}
    for i in range(num_classes):
        count = int((all_labels_arr == i).sum())
        class_sample_counts[class_labels[i]] = count
        print(f'    {class_labels[i]}: {count} samples')

    valid_class_indices = [i for i in range(num_classes) if (all_labels_arr == i).sum() > 0]
    empty_class_indices = [i for i in range(num_classes) if (all_labels_arr == i).sum() == 0]

    if empty_class_indices:
        empty_names = [class_labels[i] for i in empty_class_indices]
        print(f'  ** WARNING: test set에 샘플 0개 클래스: {empty_names} → ROC에서 제외')

    # ----------------------------------------------------------
    # 1. Confusion Matrix
    # ----------------------------------------------------------
    conf_matrix = confusion_matrix(all_labels_arr, all_preds_arr, labels=list(range(num_classes)))
    conf_matrix_normalized = np.zeros_like(conf_matrix, dtype=float)
    for i in range(num_classes):
        row_sum = conf_matrix[i].sum()
        if row_sum > 0:
            conf_matrix_normalized[i] = conf_matrix[i].astype('float') / row_sum
        else:
            conf_matrix_normalized[i] = 0.0

    # Count
    vmin = 0
    vmax = conf_matrix.max()
    tick_interval = 50
    ticks = np.arange(vmin, vmax + tick_interval, tick_interval)

    plt.figure(figsize=(6, 4))
    ax = sns.heatmap(conf_matrix, xticklabels=class_labels, yticklabels=class_labels,
                     annot=True, fmt='.0f', annot_kws={"size": 15},
                     robust=True, cmap=cmap_color, linewidth=1, square=True,
                     vmin=vmin, vmax=vmax, cbar_kws={'ticks': ticks})
    ax.set_xticklabels(labels=ax.get_xticklabels(), rotation=0, size=15)
    ax.set_yticklabels(labels=ax.get_yticklabels(), va='center', rotation=0, size=15)
    ax.set_xlabel('Predicted', size=15)
    ax.set_ylabel('True', size=15)
    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f'Confusion_matrix_Count_{task_name}.png'),
                dpi=600, transparent=True, bbox_inches='tight')
    plt.savefig(os.path.join(save_dir, f'Confusion_matrix_Count_{task_name}.eps'),
                transparent=True, bbox_inches='tight')
    plt.show()

    # Normalized
    plt.figure(figsize=(6, 4))
    ax2 = sns.heatmap(conf_matrix_normalized, xticklabels=class_labels, yticklabels=class_labels,
                      annot=True, fmt='.2f', annot_kws={"size": 15},
                      robust=True, cmap=cmap_color, linewidth=1, square=True,
                      vmin=0, vmax=1)
    ax2.set_xticklabels(labels=ax2.get_xticklabels(), rotation=0, size=15)
    ax2.set_yticklabels(labels=ax2.get_yticklabels(), va='center', rotation=0, size=15)
    ax2.set_xlabel('Predicted', size=15)
    ax2.set_ylabel('True', size=15)
    cbar2 = ax2.collections[0].colorbar
    cbar2.ax.tick_params(labelsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f'Confusion_matrix_Normalized_{task_name}.png'),
                dpi=600, transparent=True, bbox_inches='tight')
    plt.savefig(os.path.join(save_dir, f'Confusion_matrix_Normalized_{task_name}.eps'),
                transparent=True, bbox_inches='tight')
    plt.show()

    # Confusion matrix CSV
    df_cm = pd.DataFrame(conf_matrix, index=class_labels, columns=class_labels)
    df_cm.to_csv(os.path.join(save_dir, 'confusion_matrix_count.csv'), encoding='utf-8-sig')

    df_cm_norm = pd.DataFrame(conf_matrix_normalized, index=class_labels, columns=class_labels)
    df_cm_norm.to_csv(os.path.join(save_dir, 'confusion_matrix_normalized.csv'), encoding='utf-8-sig')

    # ----------------------------------------------------------
    # 2. Classification Report
    # ----------------------------------------------------------
    classification_dict = classification_report(
        all_labels_arr, all_preds_arr,
        labels=list(range(num_classes)),
        target_names=class_labels,
        output_dict=True,
        zero_division=0
    )
    classification_df = pd.DataFrame.from_dict(classification_dict).transpose().round(4)
    classification_df.to_csv(os.path.join(save_dir, 'classification_report.csv'), encoding='utf-8-sig')
    print(f'\n  Classification Report ({task_name}):')
    print(classification_df)

    # ----------------------------------------------------------
    # 3. ROC Curve & AUC
    # ----------------------------------------------------------
    all_labels_bin = label_binarize(all_labels_arr, classes=list(range(num_classes)))

    # Per-class ROC
    fpr_dict = {}
    tpr_dict = {}
    thresholds_dict = {}
    roc_auc_dict = {}

    for i in range(num_classes):
        if i in valid_class_indices:
            fpr_dict[i], tpr_dict[i], thresholds_dict[i] = roc_curve(
                all_labels_bin[:, i], all_probs_arr[:, i]
            )
            roc_auc_dict[i] = auc(fpr_dict[i], tpr_dict[i])
        else:
            fpr_dict[i] = np.array([0.0, 1.0])
            tpr_dict[i] = np.array([0.0, 1.0])
            thresholds_dict[i] = np.array([1.0, 0.0])
            roc_auc_dict[i] = np.nan

    # Micro-average (유효 클래스만)
    valid_labels_bin = all_labels_bin[:, valid_class_indices]
    valid_probs = all_probs_arr[:, valid_class_indices]

    fpr_dict["micro"], tpr_dict["micro"], _ = roc_curve(
        valid_labels_bin.ravel(), valid_probs.ravel()
    )
    roc_auc_dict["micro"] = auc(fpr_dict["micro"], tpr_dict["micro"])

    # Macro-average (유효 클래스만 보간 평균)
    all_fpr = np.unique(np.concatenate([fpr_dict[i] for i in valid_class_indices]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in valid_class_indices:
        mean_tpr += np.interp(all_fpr, fpr_dict[i], tpr_dict[i])
    mean_tpr /= len(valid_class_indices)

    fpr_dict["macro"] = all_fpr
    tpr_dict["macro"] = mean_tpr
    roc_auc_dict["macro"] = auc(fpr_dict["macro"], tpr_dict["macro"])

    # Weighted-average
    try:
        auc_weighted = roc_auc_score(
            valid_labels_bin, valid_probs,
            average='weighted', multi_class='ovr'
        )
    except Exception as e:
        auc_weighted = None
        print(f'  Weighted AUC failed: {e}')

    # ----------------------------------------------------------
    # 4. CSV - 클래스별 개별 ROC curve
    # ----------------------------------------------------------
    for i in range(num_classes):
        df_roc = pd.DataFrame({
            'FPR': fpr_dict[i],
            'TPR': tpr_dict[i],
            'Threshold': thresholds_dict[i]
        })
        if i in valid_class_indices:
            fname = f'ROC_{class_labels[i]}_AUC_{roc_auc_dict[i]:.4f}.csv'
        else:
            fname = f'ROC_{class_labels[i]}_AUC_NA_no_samples.csv'
        df_roc.to_csv(os.path.join(save_dir, fname), index=False)

    # ----------------------------------------------------------
    # 5. CSV - Micro / Macro average
    # ----------------------------------------------------------
    df_micro = pd.DataFrame({'FPR': fpr_dict["micro"], 'TPR': tpr_dict["micro"]})
    df_micro.to_csv(os.path.join(save_dir, f'ROC_micro_avg_AUC_{roc_auc_dict["micro"]:.4f}.csv'), index=False)

    df_macro = pd.DataFrame({'FPR': fpr_dict["macro"], 'TPR': tpr_dict["macro"]})
    df_macro.to_csv(os.path.join(save_dir, f'ROC_macro_avg_AUC_{roc_auc_dict["macro"]:.4f}.csv'), index=False)

    # ----------------------------------------------------------
    # 6. CSV - 통합 (OriginLab용)
    # ----------------------------------------------------------
    all_keys = valid_class_indices + ["micro", "macro"]
    max_len_roc = max(len(fpr_dict[k]) for k in all_keys)

    combined_data = {}
    for i in valid_class_indices:
        n = len(fpr_dict[i])
        combined_data[f'{class_labels[i]}_FPR'] = np.concatenate([fpr_dict[i], np.full(max_len_roc - n, np.nan)])
        combined_data[f'{class_labels[i]}_TPR'] = np.concatenate([tpr_dict[i], np.full(max_len_roc - n, np.nan)])

    for key in ["micro", "macro"]:
        n = len(fpr_dict[key])
        combined_data[f'{key}_avg_FPR'] = np.concatenate([fpr_dict[key], np.full(max_len_roc - n, np.nan)])
        combined_data[f'{key}_avg_TPR'] = np.concatenate([tpr_dict[key], np.full(max_len_roc - n, np.nan)])

    df_combined = pd.DataFrame(combined_data)
    df_combined.to_csv(os.path.join(save_dir, 'ROC_all_combined.csv'), index=False)

    # ----------------------------------------------------------
    # 7. AUC 요약
    # ----------------------------------------------------------
    auc_summary = {}
    for i in range(num_classes):
        if i in valid_class_indices:
            auc_summary[class_labels[i]] = round(roc_auc_dict[i], 4)
        else:
            auc_summary[class_labels[i]] = 'N/A (no test samples)'
    auc_summary['Micro-average'] = round(roc_auc_dict["micro"], 4)
    auc_summary['Macro-average'] = round(roc_auc_dict["macro"], 4)
    if auc_weighted is not None:
        auc_summary['Weighted-average'] = round(auc_weighted, 4)

    df_auc = pd.DataFrame.from_dict(auc_summary, orient='index', columns=['AUC'])
    df_auc.index.name = 'Class'
    df_auc.to_csv(os.path.join(save_dir, 'AUC_summary.csv'), encoding='utf-8-sig')

    print(f'\n  ===== AUC Summary ({task_name}) =====')
    print(df_auc)
    if empty_class_indices:
        print(f'  (Macro/Micro: 유효 클래스만 사용 {[class_labels[i] for i in valid_class_indices]})')
    print('  ' + '=' * 35)

    # ----------------------------------------------------------
    # 8. Softmax 확률값 전체 저장
    # ----------------------------------------------------------
    prob_columns = [f'prob_{cls}' for cls in class_labels]
    df_probs = pd.DataFrame(all_probs_arr, columns=prob_columns)
    df_probs.insert(0, 'true_label_idx', all_labels_arr)
    df_probs.insert(1, 'true_label_name', [class_labels[t] for t in all_labels_arr])
    df_probs.insert(2, 'predicted_label_idx', all_preds_arr)
    df_probs.insert(3, 'predicted_label_name', [class_labels[p] for p in all_preds_arr])
    df_probs.to_csv(os.path.join(save_dir, 'softmax_probabilities.csv'), index=False, encoding='utf-8-sig')

    # ----------------------------------------------------------
    # 9. ROC Curve Plot
    # ----------------------------------------------------------
    plt.figure(figsize=(8, 6))

    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
    for i in valid_class_indices:
        plt.plot(fpr_dict[i], tpr_dict[i], color=colors[i % len(colors)], lw=1.5,
                 label=f'{class_labels[i]} (AUC={roc_auc_dict[i]:.4f})')

    plt.plot(fpr_dict["macro"], tpr_dict["macro"], 'k--', lw=2.5,
             label=f'Macro-avg (AUC={roc_auc_dict["macro"]:.4f})')
    plt.plot(fpr_dict["micro"], tpr_dict["micro"], 'k:', lw=2.5,
             label=f'Micro-avg (AUC={roc_auc_dict["micro"]:.4f})')

    plt.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1, alpha=0.5)

    plt.xlim([-0.02, 1.02])
    plt.ylim([-0.02, 1.02])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title(f'ROC Curves - {task_full_name} ({task_name})', fontsize=14)
    plt.legend(loc='lower right', fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f'ROC_curves_{task_name}.png'),
                dpi=600, transparent=True, bbox_inches='tight')
    plt.savefig(os.path.join(save_dir, f'ROC_curves_{task_name}.eps'),
                transparent=True, bbox_inches='tight')
    plt.show()

    # ----------------------------------------------------------
    # 10. 샘플 수 요약
    # ----------------------------------------------------------
    df_samples = pd.DataFrame.from_dict(class_sample_counts, orient='index', columns=['test_samples'])
    df_samples.index.name = 'Class'
    df_samples.to_csv(os.path.join(save_dir, 'test_sample_counts.csv'), encoding='utf-8-sig')

    print(f'  All results saved to: {save_dir}')

print('\n' + '=' * 60)
print('  All tasks (D, S, F) completed!')
print('=' * 60)